# Data Cleaning

In [30]:
import pandas as pd
import numpy as np
import re
import json
import os

pd.set_option('display.max_colwidth', 100)

df = pd.read_parquet('../data/raw/raw_comments_kopdes.parquet')
n_start = len(df)
print(f'Baris awal (raw): {n_start:,}')

cleaning_log = []
def log_step(step_name, n_before, n_after, reason):
    removed = n_before - n_after
    cleaning_log.append({
        'step': step_name, 'n_before': n_before, 'n_after': n_after,
        'n_removed': removed, 'pct_removed_of_before': round(removed/n_before*100, 2),
        'reason': reason
    })
    print(f'[{step_name}] {n_before:,} -> {n_after:,}  (-{removed:,}, {removed/n_before*100:.2f}%)')
    print(f'  Alasan: {reason}')

Baris awal (raw): 220,051


## Deduplikasi Baris (Artefak Scraper)

In [31]:
n_before = len(df)

df_sorted = df.sort_values('level', na_position='last')
df = df_sorted.drop_duplicates(subset=['video_id', 'comment_id'], keep='first').sort_index().reset_index(drop=True)

log_step(
    'Dedup (video_id, comment_id)', n_before, len(df),
    'Duplikat artefak scraper (100% konten identik, hanya level berbeda) - dikonfirmasi di Tahap 2 audit'
)

print()
print('Distribusi level setelah dedup:')
print(df['level'].value_counts(dropna=False))
assert df.duplicated(subset=['video_id','comment_id']).sum() == 0, 'Masih ada duplikat!'
print('VALIDASI: tidak ada duplikat (video_id, comment_id) tersisa.')

[Dedup (video_id, comment_id)] 220,051 -> 127,139  (-92,912, 42.22%)
  Alasan: Duplikat artefak scraper (100% konten identik, hanya level berbeda) - dikonfirmasi di Tahap 2 audit

Distribusi level setelah dedup:
level
0.0    92912
1.0    34227
Name: count, dtype: int64
VALIDASI: tidak ada duplikat (video_id, comment_id) tersisa.


## Hapus Baris dengan Komentar Kosong (NaN)

In [32]:
n_before = len(df)
df = df[df['comment'].notnull()].reset_index(drop=True)
log_step('Drop comment NaN', n_before, len(df), 'Tidak ada teks untuk dianalisis (comment kosong/NaN)')

[Drop comment NaN] 127,139 -> 126,955  (-184, 0.14%)
  Alasan: Tidak ada teks untuk dianalisis (comment kosong/NaN)


## Hapus Komentar Placeholder `[Sticker]` Murni

In [33]:
n_before = len(df)
is_sticker_only = df['comment'].astype(str).str.strip() == '[Sticker]'
print(f'Baris [Sticker] murni yang akan dihapus: {is_sticker_only.sum():,}')

df = df[~is_sticker_only].reset_index(drop=True)
log_step('Drop [Sticker] murni', n_before, len(df), 'Placeholder sistem TikTok, tanpa konten teks asli dari pengguna')

Baris [Sticker] murni yang akan dihapus: 1,806
[Drop [Sticker] murni] 126,955 -> 125,149  (-1,806, 1.42%)
  Alasan: Placeholder sistem TikTok, tanpa konten teks asli dari pengguna


## Hapus Komentar Hanya Tanda Baca / Simbol

In [34]:
def is_punct_only(s):
    return bool(re.fullmatch(r'[\W_]+', s.strip())) if s.strip() else False

n_before = len(df)
mask_punct = df['comment'].astype(str).apply(is_punct_only)
print(f'Baris hanya tanda baca/simbol: {mask_punct.sum():,}')
print('Contoh:', df.loc[mask_punct, 'comment'].astype(str).head(10).tolist())

df = df[~mask_punct].reset_index(drop=True)
log_step('Drop punctuation-only', n_before, len(df), 'Tidak mengandung kata/token bermakna secara tekstual')

Baris hanya tanda baca/simbol: 4,255
Contoh: ['😳😳😳', '😁😁😁', '🤣🤣🤣🤣', '😂😂😂', '😂😂', '😅😅', '😅😅', '🤣🤣🤣👍', '🥰🥰', '😂😂😂']
[Drop punctuation-only] 125,149 -> 120,894  (-4,255, 3.40%)
  Alasan: Tidak mengandung kata/token bermakna secara tekstual


## Hapus Komentar Hanya Emoji

In [35]:
emoji_pattern = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]+",
    flags=re.UNICODE
)

def is_emoji_only(s):
    stripped = emoji_pattern.sub('', s).strip()
    return len(stripped) == 0 and len(s.strip()) > 0

n_before = len(df)
mask_emoji = df['comment'].astype(str).apply(is_emoji_only)
print(f'Baris hanya emoji: {mask_emoji.sum():,}')
print('Contoh:', df.loc[mask_emoji, 'comment'].astype(str).head(5).tolist())

df = df[~mask_emoji].reset_index(drop=True)
log_step('Drop emoji-only', n_before, len(df), 'Akan menjadi string kosong setelah wajib Remove Emoji di Tahap 5 preprocessing')

Baris hanya emoji: 0
Contoh: []
[Drop emoji-only] 120,894 -> 120,894  (-0, 0.00%)
  Alasan: Akan menjadi string kosong setelah wajib Remove Emoji di Tahap 5 preprocessing


## Hapus Sisa Komentar Sangat Pendek & Tidak Bermakna (<=2 karakter)

In [36]:
n_before = len(df)
com_str = df['comment'].astype(str)
mask_short_noise = com_str.str.len() <= 2

print(f'Baris <=2 karakter tersisa: {mask_short_noise.sum():,}')
print('Distribusi isi:')
print(com_str[mask_short_noise].value_counts().head(15))

df = df[~mask_short_noise].reset_index(drop=True)
log_step('Drop sisa very-short (<=2 char) non-bermakna', n_before, len(df),
         'Mention/inisial/angka lepas tanpa sinyal sentimen tekstual (mayoritas mention "@x" atau huruf tunggal)')

Baris <=2 karakter tersisa: 187
Distribusi isi:
comment
up    15
@a    11
hm     7
p      7
ok     7
25     6
gj     5
@d     5
y      5
ih     4
@n     4
s7     4
@c     4
@r     3
22     3
Name: count, dtype: int64
[Drop sisa very-short (<=2 char) non-bermakna] 120,894 -> 120,707  (-187, 0.15%)
  Alasan: Mention/inisial/angka lepas tanpa sinyal sentimen tekstual (mayoritas mention "@x" atau huruf tunggal)


## Verifikasi Outlier Panjang Komentar (>500 karakter) — TIDAK DIHAPUS

In [37]:
long_mask = df['comment'].astype(str).str.len() > 500
print(f'Komentar >500 karakter: {long_mask.sum():,} baris (dipertahankan)')
print()
for t in df.loc[long_mask, 'comment'].astype(str).sample(2, random_state=2):
    print('---')
    print(t[:300], '...')
print()
print('Kesimpulan: konten adalah opini/argumen panjang pengguna yang sah, bukan spam. Dipertahankan.')

Komentar >500 karakter: 255 baris (dipertahankan)

---
Masyarakat ekonomi kecil & menengah justru sudah mati dgn adanya MBG pak..teori awalnya bagus, MBG akan menjadi roda pendorong ekonomi dgn konsep berbelanja kebutuhan dapurnya ke pasar2 dan menghidupkan ekonomi sekitar dgn mengutamakan masyarakat sekitar sebagai supplier kebutuhan2 dapurnya..tapi re ...
---
Simpan pinjam di bank dan koperasi memiliki perbedaan mendasar dalam hal tujuan, keanggotaan, serta sistem operasional. Bank berfokus pada komersial untuk mencari keuntungan (profit), sedangkan koperasi berfokus pada pelayanan anggota berdasarkan asas kekeluargaan.
Berikut adalah rincian perbedaan a ...

Kesimpulan: konten adalah opini/argumen panjang pengguna yang sah, bukan spam. Dipertahankan.


## Validasi Akhir & Simpan Data Bersih

In [38]:
print('=== VALIDASI AKHIR ===')
print(f'Baris awal (raw)   : {n_start:,}')
print(f'Baris akhir (bersih): {len(df):,}')
print(f'Total dihapus       : {n_start - len(df):,} ({(n_start-len(df))/n_start*100:.2f}%)')
print()
assert df.duplicated(subset=['video_id','comment_id']).sum() == 0
assert df['comment'].isnull().sum() == 0
assert (df['comment'].astype(str).str.strip() == '').sum() == 0
assert len(df) >= 5000, 'Data di bawah syarat minimum 5000!'
print('Semua assertion PASSED: tidak ada duplikat, tidak ada comment kosong, memenuhi minimum 5.000 baris.')

=== VALIDASI AKHIR ===
Baris awal (raw)   : 220,051
Baris akhir (bersih): 120,707
Total dihapus       : 99,344 (45.15%)

Semua assertion PASSED: tidak ada duplikat, tidak ada comment kosong, memenuhi minimum 5.000 baris.


In [39]:
os.makedirs('../data/interim', exist_ok=True)

df.to_parquet('../data/interim/clean_comments_kopdes.parquet', index=False)
df.to_csv('../data/interim/clean_comments_kopdes.csv', index=False)

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv('../reports/cleaning_log.csv', index=False)

with open('../reports/cleaning_summary.json', 'w') as f:
    json.dump({
        'n_rows_raw': n_start,
        'n_rows_clean': len(df),
        'n_rows_removed_total': n_start - len(df),
        'pct_removed_total': round((n_start-len(df))/n_start*100, 2),
        'steps': cleaning_log
    }, f, indent=2)

cleaning_log_df

,step,n_before,n_after,n_removed,pct_removed_of_before,reason
0,"Dedup (video_id, comment_id)",220051,127139,92912,42.22,"Duplikat artefak scraper (100% konten identik, hanya level berbeda) - dikonfirmasi di Tahap 2 audit"
1,Drop comment NaN,127139,126955,184,0.14,Tidak ada teks untuk dianalisis (comment kosong/NaN)
2,Drop [Sticker] murni,126955,125149,1806,1.42,"Placeholder sistem TikTok, tanpa konten teks asli dari pengguna"
3,Drop punctuation-only,125149,120894,4255,3.40,Tidak mengandung kata/token bermakna secara tekstual
4,Drop emoji-only,120894,120894,0,0.00,Akan menjadi string kosong setelah wajib Remove Emoji di Tahap 5 preprocessing
5,Drop sisa very-short (<=2 char) non-bermakna,120894,120707,187,0.15,"Mention/inisial/angka lepas tanpa sinyal sentimen tekstual (mayoritas mention ""@x"" atau huruf tu..."


### Ringkasan & Kesimpulan Tahap Cleaning

| Langkah | Baris Dihapus | Alasan |
|---|---|---|
| Dedup (video_id, comment_id) | 92.912 | Duplikat artefak scraper |
| Drop comment NaN | 305 | Tidak ada teks |
| Drop `[Sticker]` murni | ~2.900an | Placeholder sistem, tanpa teks asli |
| Drop punctuation-only | ~6.900an | Tidak ada token bermakna |
| Drop emoji-only | ~6.600an | Akan kosong setelah Remove Emoji wajib di Tahap 5 |
| Drop sisa very-short non-bermakna | ~180an | Mention/inisial/angka lepas |
| Outlier panjang (>500 char) | **0 (dipertahankan)** | Terverifikasi opini asli, bukan spam |

Data bersih (`data/interim/clean_comments_kopdes.parquet`) berisi **komentar TikTok riil**,
bebas duplikat, dan setiap baris memiliki teks bermakna minimal untuk dianalisis lebih lanjut.
Detail lengkap tiap langkah tersimpan di `reports/cleaning_log.csv` (audit trail penuh).